# Tool-Using Agent



A tool-using agent decides which capability to call based on the user request, then delegates execution to the selected tool.



## Stack

- Framework: LangGraph prebuilt ReAct agent

- LLM: local Ollama via `ChatOllama(model="llama3.1:latest")`

- Pattern goal: dynamic action selection



## When to use it

- The model needs access to APIs, search, files, or data stores

- You want the LLM to choose actions dynamically



## Prerequisites

Install the local packages once in the environment used by the notebook:



```bash

pip install langchain langchain-core langgraph langchain-ollama

```

## Architecture



```mermaid

graph LR

    U[User] --> A[ReAct Agent]

    A --> T1[search_docs tool]

    A --> T2[get_weather tool]

    T1 --> A

    T2 --> A

    A --> R[Grounded Response]

```

In [ ]:
from visual_diagram_helper import render_svg_flow



nodes = [

    ("user", "User", 60, 170, "#bfdbfe"),

    ("agent", "Tool-Using Agent", 290, 170, "#bbf7d0"),

    ("search", "search_docs tool", 520, 110, "#fde68a"),

    ("weather", "get_weather tool", 520, 230, "#fde68a"),

    ("response", "Grounded Response", 980, 170, "#ddd6fe"),

]

edges = [

    ("user", "agent"),

    ("agent", "search"),

    ("agent", "weather"),

    ("search", "response"),

    ("weather", "response"),

]



render_svg_flow("Tool-Using Agent Visual Architecture", nodes, edges, width=1220, height=380)

In [5]:
from langchain.agents import create_agent

from langchain_core.tools import tool



from agent_patterns_common import build_checkpointer, get_local_llm, log_event





@tool

def search_docs(query: str) -> str:

    """Search internal notes for agent design pattern examples."""

    log_event("info", "tool_called", tool_name="search_docs", query=query)

    return f"Search results for '{query}': tool calling, routing, memory, and reflection examples."





@tool

def get_weather(city: str) -> str:

    """Get a simple weather snapshot for a city."""

    log_event("info", "tool_called", tool_name="get_weather", city=city)

    return f"Weather in {city}: 72F and sunny"





llm = get_local_llm()

memory = build_checkpointer()

agent = create_agent(model=llm, tools=[search_docs, get_weather], checkpointer=memory)

thread = {"configurable": {"thread_id": "tool-using-agent-demo"}}



log_event("info", "agent_invoke", notebook="01_tool_using_agent")

first_response = agent.invoke(

    {

        "messages": [

            (

                "system",

                "Use tools when needed. Keep responses concise and factual."

            ),

            (

                "user",

                "Find design pattern examples for AI agents and include the weather in Dubai. Return exactly 4 bullet points."

            )

        ]

    },

    config=thread,

)



second_response = agent.invoke(

    {

        "messages": [

            (

                "user",

                "Summarize the result from our previous step in exactly 2 bullet points."

            )

        ]

    },

    config=thread,

)



{

    "first_response": first_response["messages"][-1].content,

    "second_response": second_response["messages"][-1].content,

}

{"level": "INFO", "event_type": "agent_invoke", "notebook": "01_tool_using_agent"}
{"level": "INFO", "event_type": "tool_called", "tool_name": "search_docs", "query": "AI agent design patterns"}{"level": "INFO", "event_type": "tool_called", "tool_name": "get_weather", "city": "Dubai"}

{"level": "INFO", "event_type": "tool_called", "tool_name": "search_docs", "query": "AI agent design patterns examples and Dubai weather"}


{'first_response': 'Here are four AI agent design pattern examples:\n\n* **Tool Calling Pattern**: This pattern is used to encapsulate a set of tools or functions that can be called by the AI agent to perform specific tasks.\n* **Routing Pattern**: This pattern is used to manage the flow of information between different components of the AI system, such as routing user requests to the relevant AI agents.\n* **Memory Pattern**: This pattern is used to store and retrieve data in the AI system, allowing the AI agents to access and use previously learned knowledge.\n* **Reflection Pattern**: This pattern is used to enable the AI agent to inspect and modify its own behavior, allowing for self-improvement and adaptation.',
 'second_response': 'Here are 2 bullet points summarizing the result:\n\n* The AI agent design pattern examples include Tool Calling Pattern, Routing Pattern, Memory Pattern, and Reflection Pattern.\n* The current weather in Dubai is 72F and sunny.'}

## Design insight



In this pattern, the LLM owns the decision and the tools stay narrow. Keep tool schemas small and deterministic so the agent can reason about them reliably.